# Centralized convex QP reference model

This notebook builds the **centralized convex QP surrogate** used by the controller. It is not the historical benchmark replay.

The exact benchmark CTM is nonlinear/nonconvex because it contains exact `min`, `max`, positive-part clipping, merge-priority switching, and ramp queue/spillback splitting. This notebook keeps the benchmark-style objective terms but replaces the nonconvex CTM dynamics with convex linear constraints and epigraph variables.

**Role in the research pipeline**

1. `Benchmark_calculation.ipynb` gives the exact nonlinear CTM baseline and exports `shared_benchmark_inputs.pkl`.
2. This notebook loads that exported file and builds the centralized convex QP reference.
3. `ADMM_MPC.ipynb` uses the same convex QP structure inside a receding-horizon controller, then evaluates the applied commands through the exact nonlinear CTM.

**Important interpretation**

The QP is a **convex surrogate**, not an exact convex rewrite of the nonlinear benchmark. It is globally optimal for this convex model, but final traffic performance must still be evaluated in the exact CTM.


In [1]:
# 1. Load final benchmark data
import pickle, os, time
import numpy as np
import pandas as pd
import cvxpy as cp

BENCHMARK_PICKLE = "shared_benchmark_inputs.pkl"

if not os.path.exists(BENCHMARK_PICKLE):
    raise FileNotFoundError(
        "shared_benchmark_inputs.pkl was not found. Run Benchmark_calculation.ipynb first "
        "in the same folder so the final benchmark exports the shared input file."
    )

with open(BENCHMARK_PICKLE, "rb") as f:
    S = pickle.load(f)

cells = [f"Cell {i}" for i in range(1, 10)]
ramps = list(S["ramp_ids"])
T = int(S["num_steps"])
dt = float(S["delta_t"])
arrival_multiplier = float(S["arrival_multiplier"])

mu = np.array([float(S["movement_factor_by_cell"][c]) for c in cells])
w = np.array([float(S["wave_speed_ratio_by_cell"][c]) for c in cells])
Cin = np.array([float(S["inflow_capacity"][c]) for c in cells])
Cout = np.array([float(S["outflow_capacity"][c]) for c in cells])
Xmax = np.array([float(S["physical_capacity"][c]) for c in cells])
Xsafe = np.array([float(S["safe_threshold_capacity"][c]) for c in cells])
beta = np.array([float(S["exit_split_by_cell"][c]) for c in cells])
tau = np.array([float(S["tt_ff_min"][c]) for c in cells])

e = np.array([[float(S["external_inflow_series"][k][c]) for k in range(T)] for c in cells])
f = np.array([[float(S["fixed_outflow_series"][k][c]) for k in range(T)] for c in cells])

a = np.array([[float(S["ramp_arrival_series"][r][k]) for k in range(T)] for r in ramps])
observed_release = np.array([[float(S["observed_release_series"][r][k]) for k in range(T)] for r in ramps])

# Fair metering-authority cap. This matches the final ADMM-MPC setup.
METER = arrival_multiplier
u_max = METER * observed_release

d = np.array([float(S["q_in_boundary_series"][k]) for k in range(T)])
x0 = np.array([float(S["mainline_initial_state"][c]) for c in cells])

ramp_queue_0 = S.get("ramp_queue_0", {})
external_queue_0 = S.get("external_queue_0", {})
W0 = np.array([float(ramp_queue_0.get(r, 0.0)) + float(external_queue_0.get(r, 0.0)) for r in ramps])
U0 = float(S.get("upstream_queue_0", S.get("boundary_queue_0", 0.0)))

Rmax = np.array([float(S["ramp_max_queue_by_u"][r]) for r in ramps])
gamma = float(S["gamma"])
l1 = float(S["lambda_1"])
l2 = float(S["lambda_2"])
l3 = float(S["lambda_3"])
l4 = float(S["lambda_4"])

# Official centralized convex QP reference uses the benchmark objective weight.
MAINLINE_WEIGHT = 1.0

cidx = {c: i for i, c in enumerate(cells)}
Minc = np.zeros((9, len(ramps)))
for j, r in enumerate(ramps):
    Minc[cidx[S["ramp_cell_map"][r]], j] = 1.0

bt = S["official_totals"]
if float(bt["raw_objective"]) > 1_000_000:
    raise RuntimeError(
        "The loaded benchmark pickle appears to use the old quadratic-spillback objective. "
        "Re-run the final linear-spillback Benchmark_calculation notebook first."
    )

print(f"loaded {BENCHMARK_PICKLE}")
print(f"cells=9 | ramps={len(ramps)} | T={T} | arrival_multiplier={arrival_multiplier} | METER={METER}")
print(f"benchmark raw objective={bt['raw_objective']:.6f}")


loaded shared_benchmark_inputs.pkl
cells=9 | ramps=4 | T=480 | arrival_multiplier=1.2 | METER=1.2
benchmark raw objective=130422.809368


## 2. The convex QP

The QP replaces exact nonconvex CTM equalities with convex constraints.

### Exact benchmark idea

The exact model uses formulas such as

$
q = \min(	ext_{sending},	ext_{receiving}).
$

That equality is nonconvex when the flow and state variables are decision variables.

### Convex replacement

The QP instead imposes upper bounds:

$
q \le 	ext{sending},
\qquad
q \le 	ext{receiving}.
$

The optimizer then chooses flows and ramp releases that minimize the convex objective while respecting those bounds.

### Key variables

- $(x_{i,k})$: mainline occupancy.
- $(L_{i,k})$: total leaving flow from Cell \(i\).
- $(v_{m,k})$: accepted ramp release decision in the convex surrogate.
- $(W_{m,k})$: total ramp-side waiting demand.
- $(B_{m,k})$: spillback epigraph variable.
- $(U_k)$: upstream boundary queue.
- $(d_{i,k})$, $(s^{door})$, $(s^{safe})$, $(s^{phys})$: epigraph variables for clipped delay and penalties.

### Important fair-bound constraint

The QP uses the same metering-authority rule as the final controller:

$
0 \le v_{m,k} \le u^{\max}_{m,k},
\qquad
u^{\max}_{m,k}=	ext{arrival\_multiplier}\cdot u^{obs}_{m,k}.
$

This prevents the centralized convex QP from receiving more ramp-release authority than ADMM-MPC or the fixed-policy comparison.

### Objective

The objective keeps the benchmark-style cost terms:

$
D_{mainline}+D_{ramp}+F_{fairness}+P_{doorway}+P_{safe}+P_{physical}+P_{spillback}.
$

The spillback penalty is linear in the final setup:

$
P_{spillback}=\lambda_4\sum_{m,k} B_{m,k}.
$

Fairness uses total-queue stress \(W/R^{max}\) rather than physical-queue stress. This is a convex surrogate choice that prevents the optimizer from gaming the physical/spillback split.


In [2]:
# 3. Build the centralized convex QP
nR = len(ramps)

x = cp.Variable((9, T + 1), nonneg=True)
L = cp.Variable((9, T), nonneg=True)
v = cp.Variable((nR, T), nonneg=True)
W = cp.Variable((nR, T + 1), nonneg=True)
Bv = cp.Variable((nR, T), nonneg=True)
U = cp.Variable(T + 1, nonneg=True)
qin1 = cp.Variable(T, nonneg=True)
ftil = cp.Variable((9, T), nonneg=True)

dly = cp.Variable((9, T), nonneg=True)
ss = cp.Variable((9, T), nonneg=True)
sp = cp.Variable((9, T), nonneg=True)
sd = cp.Variable((9, T), nonneg=True)

qd = cp.multiply((1.0 - beta)[:, None], L)
inflow = cp.vstack([cp.reshape(qin1, (1, T), order="C"), qd[0:8, :]]) + e + Minc @ v

constraints = []
constraints += [x[:, 0] == x0]
constraints += [W[:, 0] == W0]
constraints += [U[0] == U0]

# Sending limits.
constraints += [L <= cp.multiply(mu[:, None], x[:, :T])]
constraints += [L <= Cout[:, None]]

# Receiving and merge-capacity limits, with external inflow included in inflow.
constraints += [inflow <= Cin[:, None]]
constraints += [inflow <= cp.multiply(w[:, None], Xmax[:, None] - x[:, :T])]

# Fixed local outflow clipping and mainline conservation.
constraints += [ftil <= f]
constraints += [x[:, 1:] == x[:, :T] + inflow - L - ftil]

# Ramp demand, fair command authority, total queue conservation, and spillback epigraph.
constraints += [v <= W[:, :T] + a]
constraints += [v <= u_max]
constraints += [W[:, 1:] == W[:, :T] + a - v]
constraints += [Bv >= W[:, 1:] - Rmax[:, None]]

# Boundary inflow and queue conservation.
constraints += [qin1 <= U[:T] + d]
constraints += [U[1:] == U[:T] + d - qin1]

# Epigraphs for clipped mainline delay and capacity penalties.
constraints += [dly >= (x[:, :T] + x[:, 1:]) / 2.0 * dt - cp.multiply(tau[:, None], L + ftil)]
constraints += [ss >= x[:, 1:] - Xsafe[:, None]]
constraints += [sp >= x[:, 1:] - Xmax[:, None]]
constraints += [sd >= inflow - Cin[:, None]]

mainline = MAINLINE_WEIGHT * (cp.sum(dly) + dt * cp.sum((U[:T] + U[1:]) / 2.0))
ramp = dt * cp.sum((W[:, :T] + W[:, 1:]) / 2.0)

fair = 0
for m in range(nR):
    for n in range(m + 1, nR):
        fair += cp.sum_squares(W[m, 1:] / Rmax[m] - W[n, 1:] / Rmax[n])

objective = cp.Minimize(
    mainline
    + ramp
    + gamma * fair
    + l1 * cp.sum_squares(sd)
    + l2 * cp.sum_squares(ss)
    + l3 * cp.sum_squares(sp)
    + l4 * cp.sum(Bv)
)

prob = cp.Problem(objective, constraints)

print("QP variables:", sum(z.size for z in [x, L, v, W, Bv, U, qin1, ftil, dly, ss, sp, sd]))
print("DCP:", prob.is_dcp(), "| QP:", prob.is_qp())
assert prob.is_dcp(), "The centralized convex QP is not DCP."
assert prob.is_qp(), "The centralized convex model is not recognized as a QP."


QP variables: 36974
DCP: True | QP: True


In [3]:
# 4. Solve to global optimum of the convex QP surrogate
available_solvers = set(cp.installed_solvers())
if "CLARABEL" in available_solvers:
    solver = cp.CLARABEL
elif "OSQP" in available_solvers:
    solver = cp.OSQP
else:
    raise RuntimeError(f"No supported QP solver found. Installed solvers: {sorted(available_solvers)}")

t0 = time.time()
prob.solve(solver=solver, warm_start=True)
elapsed = time.time() - t0

print("status:", prob.status, "| solve %.2fs" % elapsed, "| solver:", solver)
print("QP objective (= centralized convex surrogate objective):", round(float(prob.value), 6))

if prob.status not in {"optimal", "optimal_inaccurate"}:
    raise RuntimeError(f"QP solve failed with status: {prob.status}")


status: optimal | solve 0.65s | solver: CLARABEL
QP objective (= centralized convex surrogate objective): 52660.107031


In [4]:
# 5. Compare against benchmark and best fixed policy using complete objective accounting
if any(z.value is None for z in [x, L, v, W, Bv, U, ftil, dly, ss, sp, sd]):
    raise RuntimeError("Solve the QP before running the comparison cell.")

xv = np.asarray(x.value)
Lv = np.asarray(L.value)
Wv = np.asarray(W.value)
Bvv = np.asarray(Bv.value)
Uv = np.asarray(U.value)
ftv = np.asarray(ftil.value)

# QP objective components from epigraph variables. These are the quantities actually optimized.
qp_mainline = float(np.sum(dly.value) + dt * np.sum((Uv[:T] + Uv[1:]) / 2.0))
qp_ramp = float(np.sum((Wv[:, :T] + Wv[:, 1:]) / 2.0) * dt)
qp_fairness = float(gamma * sum(
    np.sum((Wv[m, 1:] / Rmax[m] - Wv[n, 1:] / Rmax[n]) ** 2)
    for m in range(nR)
    for n in range(m + 1, nR)
))
qp_doorway = float(l1 * np.sum(np.asarray(sd.value) ** 2))
qp_safe = float(l2 * np.sum(np.asarray(ss.value) ** 2))
qp_physical = float(l3 * np.sum(np.asarray(sp.value) ** 2))
qp_spillback = float(l4 * np.sum(Bvv))
qp_raw = qp_mainline + qp_ramp + qp_fairness + qp_doorway + qp_safe + qp_physical + qp_spillback

# Additional diagnostics from the actual affine expressions. These should be close to the epigraph values.
inflow_val = np.asarray(inflow.value)
expr_mainline = float(
    np.sum(np.maximum((xv[:, :T] + xv[:, 1:]) / 2.0 * dt - (Lv + ftv) * tau[:, None], 0.0))
    + np.sum((Uv[:T] + Uv[1:]) / 2.0 * dt)
)
expr_doorway = float(l1 * np.sum(np.maximum(inflow_val - Cin[:, None], 0.0) ** 2))
expr_safe = float(l2 * np.sum(np.maximum(xv[:, 1:] - Xsafe[:, None], 0.0) ** 2))
expr_physical = float(l3 * np.sum(np.maximum(xv[:, 1:] - Xmax[:, None], 0.0) ** 2))

bt = S["official_totals"]

# Best fixed-policy result from the final linear-spillback fixed-policy sweep.
# This row must match fixed_policy.ipynb under arrival_multiplier=1.2 and release_scale=1.10.
fixed_best = {
    "policy": "Best fixed policy, scale 1.10",
    "mainline": 38399.822181,
    "ramp": 23011.623185,
    "fairness": 214.095980,
    "doorway": 0.0,
    "safe": 1766.283775,
    "physical": 0.0,
    "spillback": 20044.591946,
    "raw": 83436.417067,
}

rows = [
    {
        "policy": "Benchmark observed commands",
        "mainline": float(bt["mainline_delay"]),
        "ramp": float(bt["local_delay"]),
        "fairness": float(bt["fairness_penalty"]),
        "doorway": float(bt.get("doorway_penalty", 0.0)),
        "safe": float(bt.get("safe_penalty", bt.get("safe_occupancy_penalty", 0.0))),
        "physical": float(bt.get("physical_penalty", bt.get("physical_capacity_penalty", 0.0))),
        "spillback": float(bt["spillback_penalty"]),
        "raw": float(bt["raw_objective"]),
    },
    fixed_best,
    {
        "policy": "Centralized convex QP reference",
        "mainline": qp_mainline,
        "ramp": qp_ramp,
        "fairness": qp_fairness,
        "doorway": qp_doorway,
        "safe": qp_safe,
        "physical": qp_physical,
        "spillback": qp_spillback,
        "raw": qp_raw,
    },
]

comparison = pd.DataFrame(rows)
display(comparison.round(3))

print("QP objective component sum:", round(qp_raw, 6))
print("CVXPY reported objective:", round(float(prob.value), 6))
print("absolute objective mismatch:", abs(qp_raw - float(prob.value)))
print("QP raw improvement vs benchmark: %.3f%%" % (100.0 * (1.0 - qp_raw / float(bt["raw_objective"]))))
print("QP raw improvement vs best fixed policy: %.3f%%" % (100.0 * (1.0 - qp_raw / fixed_best["raw"])))

print("epigraph diagnostic | mainline:", round(abs(qp_mainline - expr_mainline), 6))
print("epigraph diagnostic | doorway:", round(abs(qp_doorway - expr_doorway), 6))
print("epigraph diagnostic | safe:", round(abs(qp_safe - expr_safe), 6))
print("epigraph diagnostic | physical:", round(abs(qp_physical - expr_physical), 6))

if abs(qp_raw - float(prob.value)) > 1e-3:
    raise RuntimeError("The comparison table does not match the QP objective. Check objective accounting.")


,policy,mainline,ramp,fairness,doorway,safe,physical,spillback,raw
0,Benchmark observed commands,25744.199,44806.730,143.624,0.0,0.000,0.0,59728.257,130422.809
1,"Best fixed policy, scale 1.10",38399.822,23011.623,214.096,0.0,1766.284,0.0,20044.592,83436.417
2,Centralized convex QP reference,50895.983,1614.704,147.446,0.0,0.805,0.0,1.170,52660.107


QP objective component sum: 52660.107031
CVXPY reported objective: 52660.107031
absolute objective mismatch: 0.0
QP raw improvement vs benchmark: 59.624%
QP raw improvement vs best fixed policy: 36.886%
epigraph diagnostic | mainline: 2.7e-05
epigraph diagnostic | doorway: 1.8e-05
epigraph diagnostic | safe: 1.6e-05
epigraph diagnostic | physical: 1.6e-05


## Final notes

- This notebook is the **centralized convex QP reference**, not the exact historical benchmark.
- The exact benchmark remains `Benchmark_calculation.ipynb`.
- The QP uses the same fair metering-authority cap as the final MPC controller: `v <= arrival_multiplier * observed_release`.
- The comparison table now includes all objective terms: mainline, ramp, fairness, doorway, safe, physical, and linear spillback.
- The LP row was removed because the LP is no longer part of the final four-file research setup.
- The fixed-policy row now matches the final fixed-policy sweep: release scale 1.10 under `arrival_multiplier = 1.2`.
- Final controller performance should still be evaluated by applying commands through the exact nonlinear CTM, as done in `ADMM_MPC.ipynb`.
